# Robot Modeling

## variable setup

In [38]:
import sympy as sp
import numpy as np
from sympy.physics.mechanics import dynamicsymbols
from sympy.physics.mechanics import ReferenceFrame, Vector, inertia
from IPython.display import display, Math


# 1. Define the time symbol (the independent variable)
t = sp.Symbol('t')
#manually changed here as I decided to use vectors instead of symbols
#link_chars = ["a_"+str(i) for i in range(1,5)]
#link_symbols = [Vector(' '.join(link_chars),) for i in range(1,5)] 
# modify this code so that i get 5 vector names following the format (a_1) 
# and then populate the 5 vectors with sympy symbols that have the form (a_1_x,a_1_y,a_1_z)
link_chars = ["a_"+str(i) for i in range(1,5)]
vectors = [sp.Matrix([* sp.symbols(f'a_{i}_x a_{i}_y a_{i}_z')])  for i in range(1, 7)]

link_mass = ["M_"+str(i) for i in range(1,5)]
masses = sp.Matrix([sp.Symbol(f"M{i}") for i in range(1, 7)])
display(Math(sp.latex(vectors)))

#in this case a is the distance from CoM to rotation axisbg
a_1,a_2,a_3,a_4,a_5,a_6 = vectors[0:6]
functions = sp.Matrix([sp.Function(rf'\theta_'+str(i))(t) for i in range(1, 7)])
functions_distance = sp.Matrix([sp.Function(rf'd_'+str(i))(t) for i in range(1, 7)])
# 4. Create time-varying functions for the joints
# Unpack for easy access
#theta_a, theta_b, theta_c, theta_d, theta_e, theta_f = thetas
display(Math(sp.latex(functions.T)))
theta_1,theta_2,theta_3,theta_4,theta_5,theta_6 = functions[0:6]
#not needed for this calc but adding prismatic functions anyways
d_1,d_2,d_3,d_4,d_5,d_6 = functions_distance
inertias = ([sp.Matrix([
    * [sp.symbols(f"^{{CM}}I_{{xx}}^{i} ^{{CM}}I_{{xy}}^{i} ^{{CM}}I_{{xz}}^{i}")],
    * [sp.symbols(f"^{{CM}}I_{{xy}}^{i} ^{{CM}}I_{{yy}}^{i} ^{{CM}}I_{{yz}}^{i}")],
    * [sp.symbols(f"^{{CM}}I_{{xz}}^{i} ^{{CM}}I_{{yz}}^{i} ^{{CM}}I_{{zz}}^{i}")]

])for i in range(1,7)])
# To view the first matrix formatted as Math:
display(Math(sp.latex(inertias)))
Icm_1,Icm_2,Icm_3,Icm_4,Icm_5,Icm_6 = inertias[0:6]
display(Math(sp.latex(masses.T)))
m_1,m_2,m_3,m_4,m_5,m_6 = masses


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [39]:
#display(Math(sp.latex(
#    [masses[i]*(vectors[i]*vectors[i].T) 
#     for i in range(0,len(masses))])))

parallel_axis_tensor =  [inertias[i]+masses[i]*(vectors[i]*vectors[i].T) 
     for i in range(0,len(masses))]
display(Math(
    sp.latex(sp.Symbol("^{CM}I^i")) + "+" + sp.latex(sp.Symbol("M_i"))  + 
    sp.latex(sp.Symbol(r"\hat{r_i}^2")) 
             ))
display(Math(
    sp.latex([inertias[0]])+"+"+sp.latex([masses[0]])+ sp.latex(vectors[0])+ sp.latex(vectors[0].T)
    ))
 
display(Math(sp.latex(parallel_axis_tensor
   )))



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Moment of inertia

## DH tables to foward kinematics


In [40]:
#import DH_calculator as calc
## temp fix
import sympy as sp

def get_dh_matrix(theta, d, a, alpha):
    """
    Calculates the homogeneous transformation matrix using DH parameters.
    Order: Rot_z(theta) -> Trans_z(d) -> Trans_x(a) -> Rot_x(alpha)
    """
    
    # 1. Rotation about the Z-axis by theta
    Rz = sp.Matrix([
        [sp.cos(theta), -sp.sin(theta), 0, 0],
        [sp.sin(theta),  sp.cos(theta), 0, 0],
        [0,              0,             1, 0],
        [0,              0,             0, 1]
    ])
    
    # 2. Translation along the Z-axis by d
    Tz = sp.Matrix([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, d],
        [0, 0, 0, 1]
    ])
    
    # 3. Translation along the X-axis by a
    Tx = sp.Matrix([
        [1, 0, 0, a],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ])
    
    # 4. Rotation about the X-axis by alpha
    Rx = sp.Matrix([
        [1, 0,            0,             0],
        [0, sp.cos(alpha), -sp.sin(alpha), 0],
        [0, sp.sin(alpha),  sp.cos(alpha), 0],
        [0, 0,            0,             1]
    ])
    
    # Multiply the transformations in order
    # T = (Rz * Tz) * (Tx * Rx)
    T = Rz * Tz * Tx * Rx
    
    return sp.simplify(T)
sp.Matrix([
    [theta_1,a_1[2],a_1[0],0],
    [theta_2,a_2[2],a_2[0],0],
    [theta_3,a_3[2],a_3[0],0],
    [theta_4,a_4[2],a_4[0],0],
    [theta_5,a_5[2],a_5[0],0],
    [theta_6,a_6[2],a_6[0],0],
    ])
tf_matrixes = [get_dh_matrix(functions[i],
                    functions_distance[i],
                    0,
                   0) for i in range(0,len(masses))]
display(Math(sp.latex(tf_matrixes)))



<IPython.core.display.Math object>

In [41]:
offset_distance_x = [sp.Symbol(f'a_{i}') for i in range(1,len(functions)+1)]
offset_distance_z = [sp.Symbol(f'd_{i}') for i in range(1,len(functions)+1)]
offset_angle_x_axis = [-sp.pi/2,-sp.pi/2,0,sp.pi/2,-sp.pi/2,0]
offset_angle_z_axis = [0,sp.pi/2,0,0,0,0]

dh_param_fencebot = sp.Matrix(
    [[functions[i]+offset_angle_z_axis[i],
      offset_distance_z[i],
      offset_distance_x[i],
      offset_angle_x_axis[i]] for i in range(0,len(functions))])

display(Math(sp.latex(dh_param_fencebot)))
tf_matrixes_fencebot = [get_dh_matrix(dh_param_fencebot[i,0] ,
                        dh_param_fencebot[i,1],dh_param_fencebot[i,2],dh_param_fencebot[i,3]
)for i in range(0,len(functions))]


<IPython.core.display.Math object>

## Foward Kinematics

In [42]:
tf_0_i = sp.eye(4)
for i in tf_matrixes:
    tf_0_i *= i

tf_0_4 = sp.eye(4)
for i in range(0,4):
    tf_0_4 *= tf_matrixes[i]

display(Math(sp.latex(
    sp.simplify(tf_0_i)
    )))
display(Math(sp.latex(
    sp.simplify(tf_0_4)
    )))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Foward kinematics Fencebot


In [43]:
tf_0_i_fencebot= sp.eye(4)
for i in tf_matrixes_fencebot:
    tf_0_i_fencebot *= i

tf_0_4_fencebot = sp.eye(4)
for i in range(0,4):
    tf_0_4_fencebot *= tf_matrixes_fencebot[i]

display(Math(sp.latex(
    sp.simplify(tf_0_i_fencebot)
    )))
display(Math(sp.latex(
    sp.simplify(tf_0_4_fencebot)
    )))

fencebot_fk = tf_0_i_fencebot[:,3]
display(Math(sp.latex(fencebot_fk)))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
functions_dot = [sp.Symbol(f"\\dot {{\\theta}}_{i}") for i in range(1,7)]
display(Math(sp.latex(functions_dot)))
fencebot_fk_derivative = fencebot_fk.diff(t)
for i in range(0,len(functions)):
    fencebot_fk_derivative = fencebot_fk_derivative.subs(functions[i].diff(t),functions_dot[i])
display(Math(sp.latex(sp.simplify(fencebot_fk_derivative))))
jacobian = [[position_function.diff(functions_dot[0]),
    position_function.diff(functions_dot[1]),
    position_function.diff(functions_dot[2]),
    position_function.diff(functions_dot[3]),
    position_function.diff(functions_dot[4]),
    position_function.diff(functions_dot[5])
            ]for position_function in fencebot_fk_derivative ]


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Inverse Kinematics

In [45]:
x = 1
y = 1
z = 1
import math
## analytical sln 1,sln2
theta_1 = [math.atan2(y,z),-math.atan2(y,z)]